In [153]:
import pandas as pd
import numpy as np
from typing import List, Tuple

In [206]:
df = pd.read_csv('../data/lab04/q02.csv')

df.head()

for _, subset in df.groupby('AGE'):
  print(subset)

   ID     AGE  JOB_STATUS  OWNS_HOUSE CREDIT_RATING CLASS
5   6  Middle       False       False          Fair    No
6   7  Middle       False       False          Good    No
7   8  Middle        True        True          Good   Yes
8   9  Middle       False        True     Excellent   Yes
9  10  Middle       False        True     Excellent   Yes
    ID  AGE  JOB_STATUS  OWNS_HOUSE CREDIT_RATING CLASS
10  11  Old       False        True     Excellent   Yes
11  12  Old       False        True          Good   Yes
12  13  Old        True       False          Good   Yes
13  14  Old        True       False     Excellent   Yes
14  15  Old       False       False          Fair    No
   ID    AGE  JOB_STATUS  OWNS_HOUSE CREDIT_RATING CLASS
0   1  Young       False       False          Fair    No
1   2  Young       False       False          Good    No
2   3  Young        True       False          Good   Yes
3   4  Young        True        True          Fair   Yes
4   5  Young       False       

In [207]:
from dataclasses import dataclass

@dataclass
class Node:
  name: str
  options: List[Tuple[str, 'Node']]

def entropy(x: pd.DataFrame | pd.Series):
  y = x.value_counts(normalize=True)
  return -(y * np.log2(y)).sum()

def create_decision_tree(X: pd.DataFrame, y: pd.Series):
  if len(y.unique()) == 1:
    return Node(name=y.iloc[0], options=[])

  N = len(X)
  base_entropy = entropy(y)
  
  best_col = (0, None)
  for col in X.columns:
    gain = base_entropy - sum(len(subset) / N * entropy(y.loc[subset.index]) for _, subset in X.groupby(col))
    best_col = max(best_col, (gain, col), key=lambda k: k[0])

  best_col = best_col[1]
  if best_col is None:
    return Node(name=y.mode()[0], options=[])

  options = []
  for val, subset in X.groupby(best_col):
    selected = subset.drop(columns=[best_col])
    child = create_decision_tree(selected, y.loc[selected.index])
    options.append((val, child))

  node = Node(name=best_col, options=options)
  return node

def dfs(tree: Node, height: int = 1):
  print("\t" * (height - 1), tree.name, " (END)" if not tree.options else "", sep="")

  for option, node in tree.options:
    print("\t" * height, option, sep="")
    if node:
      dfs(node, height+1)

tree = create_decision_tree(df.drop(columns=['ID', 'CLASS']), df['CLASS'])

dfs(tree)

OWNS_HOUSE
	False
	JOB_STATUS
		False
		No (END)
		True
		Yes (END)
	True
	Yes (END)
